# ShrimpDiseaseImageBD — Full Dataset EDA, Label Audit, Visualization, and Model-result Analysis

**Goal:** analyze the current ShrimpDiseaseImageBD dataset for object detection and later classification-gate design.

This notebook does **not** train models. It audits:

1. Dataset structure from KaggleHub.
2. Raw image classification folders: `Healthy`, `BG`, `WSSV`, `BG_WSSV`.
3. Annotated disease folders: `1. BG`, `2. WSSV`, `4. WSSV_BG`.
4. YOLO label validity and canonical remapping:
   - `0 = BG`
   - `1 = WSSV`
5. Box statistics, small-object risk, class imbalance, image sizes, file sizes.
6. Visualizations:
   - raw class contact sheets,
   - annotated overlay samples,
   - bbox-size distributions,
   - box-center heatmaps,
   - box count per image,
   - full-dataset thumbnail contact sheets saved to disk.
7. Optional model-result analysis if `final_model_comparison.csv` exists from previous YOLO benchmark.

Protocol note:
- Healthy is analyzed as image-level class.
- Healthy is **not** used as object-detection class because it has no disease-region bounding boxes.

In [1]:
from pathlib import Path
import os, sys, json, csv, math, random, shutil, statistics, platform, traceback
from datetime import datetime
from collections import defaultdict, Counter

WORKDIR = Path('/home/drnguyenvinh/notebooks/shrimp_dataset_eda_label_model_analysis_v1')
WORKDIR.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET = 'nhanayai/shrimpdiseaseimagebd'
LOCAL_RAW = WORKDIR / 'kaggle_raw' / 'shrimpdiseaseimagebd'
EDA_DIR = WORKDIR / 'eda_outputs'
FIG_DIR = EDA_DIR / 'figures'
TABLE_DIR = EDA_DIR / 'tables'
GALLERY_DIR = EDA_DIR / 'galleries'
OVERLAY_DIR = EDA_DIR / 'overlays'

for p in [LOCAL_RAW, EDA_DIR, FIG_DIR, TABLE_DIR, GALLERY_DIR, OVERLAY_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
IMG_EXTS = {'.jpg','.jpeg','.png','.bmp','.webp','.JPG','.JPEG','.PNG','.BMP','.WEBP'}

print('WORKDIR:', WORKDIR)
print('LOCAL_RAW:', LOCAL_RAW)
print('EDA_DIR:', EDA_DIR)
print('Python:', sys.version)
print('Platform:', platform.platform())

WORKDIR: /home/drnguyenvinh/notebooks/shrimp_dataset_eda_label_model_analysis_v1
LOCAL_RAW: /home/drnguyenvinh/notebooks/shrimp_dataset_eda_label_model_analysis_v1/kaggle_raw/shrimpdiseaseimagebd
EDA_DIR: /home/drnguyenvinh/notebooks/shrimp_dataset_eda_label_model_analysis_v1/eda_outputs
Python: 3.13.2 | packaged by Anaconda, Inc. | (main, Feb  6 2025, 18:56:02) [GCC 11.2.0]
Platform: Linux-6.14.0-37-generic-x86_64-with-glibc2.39


## 1. Minimal dependency check

This notebook installs only `kagglehub` if missing. It does not upgrade Torch, NumPy, OpenCV, Pandas, Pillow, Matplotlib, or Scikit-learn.

In [2]:
import importlib.util, subprocess

def is_installed(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None

def pip_install_minimal(pkgs):
    if not pkgs:
        print('Required packages already installed.')
        return
    cmd = [sys.executable, '-m', 'pip', 'install'] + pkgs
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

missing = []
if not is_installed('kagglehub'):
    missing.append('kagglehub')
pip_install_minimal(missing)

try:
    import pandas as pd
except Exception:
    pd = None
    print('WARNING: pandas unavailable. Tables will be written with csv only.')

try:
    import numpy as np
except Exception:
    np = None
    print('WARNING: numpy unavailable. Some numeric summaries will be limited.')

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None
    print('WARNING: matplotlib unavailable. Plots will be skipped.')

from PIL import Image, ImageDraw, ImageFont
print('pandas:', getattr(pd, '__version__', None))
print('numpy:', getattr(np, '__version__', None))

Required packages already installed.
pandas: 3.0.3
numpy: 2.4.6


## 2. Download dataset using KaggleHub

In [3]:
import kagglehub

dataset_cache_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
print('KaggleHub dataset path:', dataset_cache_path)

if not any(LOCAL_RAW.iterdir()):
    print('Copying dataset into project workspace...')
    shutil.copytree(dataset_cache_path, LOCAL_RAW, dirs_exist_ok=True)
else:
    print('Dataset already copied:', LOCAL_RAW)

# Quick tree preview
def print_tree(root, max_depth=4):
    root = Path(root)
    root_depth = len(root.parts)
    for p in sorted(root.rglob('*')):
        depth = len(p.parts) - root_depth
        if depth > max_depth:
            continue
        indent = '  ' * depth
        print(f'{indent}[D] {p.name}' if p.is_dir() else f'{indent}[F] {p.name}')

print('Dataset root:', LOCAL_RAW)
print_tree(LOCAL_RAW, max_depth=3)

/opt/miniconda3/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


KaggleHub dataset path: /home/drnguyenvinh/.cache/kagglehub/datasets/nhanayai/shrimpdiseaseimagebd/versions/1
Copying dataset into project workspace...
Dataset root: /home/drnguyenvinh/notebooks/shrimp_dataset_eda_label_model_analysis_v1/kaggle_raw/shrimpdiseaseimagebd
  [D] ShrimpDiseaseImageBD An Image Dataset for Computer Vision-Based Detection of Shrimp Diseases in Bangladesh
    [D] Root
      [D] Annotated Diseased Shrimp Images
      [D] Raw Images
      [F] Readme.docx
      [F] ShrimpDiseaseBD_Summary.xlsx


## 3. Helper functions: locating folders and reading YOLO labels

In [4]:
def norm_name(s: str) -> str:
    return (str(s).lower().replace(' ', '').replace('_', '').replace('-', '')
            .replace('.', '').replace('(', '').replace(')', ''))

def list_images(p):
    if p is None:
        return []
    p = Path(p)
    if not p.exists():
        return []
    return sorted([x for x in p.iterdir() if x.is_file() and x.suffix in IMG_EXTS])

def list_label_txts(p):
    if p is None:
        return []
    p = Path(p)
    if not p.exists():
        return []
    return sorted([x for x in p.iterdir() if x.is_file() and x.suffix.lower() == '.txt'])

def find_child_dir(parent: Path, wanted_names):
    wanted = {norm_name(x) for x in wanted_names}
    for child in Path(parent).iterdir():
        if child.is_dir() and norm_name(child.name) in wanted:
            return child
    return None

def raw_class_key_from_name(name: str):
    n = norm_name(name)
    if n == 'healthy': return 'Healthy'
    if n in {'bg','blackgill'}: return 'BG'
    if n in {'wssv','whitespot','whitespotsyndromevirus'}: return 'WSSV'
    if n in {'bgwssv','wssvbg'}: return 'BG_WSSV'
    return None

def ann_group_key_from_name(name: str):
    n = norm_name(name)
    if n in {'1bg','bg','blackgill'}: return 'BG'
    if n in {'2wssv','wssv','whitespot','whitespotsyndromevirus'}: return 'WSSV'
    if n in {'4wssvbg','wssvbg','bgwssv'}: return 'WSSV_BG'
    return None

def remap_label(source_group: str, source_cls: int) -> int:
    # canonical OD classes: 0 = BG, 1 = WSSV
    if source_group == 'BG':
        if source_cls != 0: raise ValueError(f'Unexpected source class in BG folder: {source_cls}')
        return 0
    if source_group == 'WSSV':
        if source_cls != 0: raise ValueError(f'Unexpected source class in WSSV folder: {source_cls}')
        return 1
    if source_group == 'WSSV_BG':
        if source_cls == 0: return 1  # WSSV-like region
        if source_cls == 1: return 0  # BG-like region
    raise ValueError(f'Unknown label mapping: {source_group=} {source_cls=}')

def image_level_from_label_set(label_set):
    if 0 in label_set and 1 in label_set: return 'WSSV_BG'
    if 0 in label_set: return 'BG'
    if 1 in label_set: return 'WSSV'
    return 'NO_BOX'

def read_image_size(path):
    with Image.open(path) as im:
        return im.size

## 4. Locate raw classification folders and annotated detection folders

In [ ]:
raw_class_dirs = {}
for d in LOCAL_RAW.rglob('*'):
    if not d.is_dir():
        continue
    key = raw_class_key_from_name(d.name)
    if key:
        imgs = list_images(d)
        if imgs:
            old = raw_class_dirs.get(key)
            if old is None or len(imgs) > old['n_images']:
                raw_class_dirs[key] = {'dir': d, 'n_images': len(imgs)}

print('Raw class folders:')
for k in ['Healthy','BG','WSSV','BG_WSSV']:
    print(k, '=>', raw_class_dirs.get(k))

ann_group_dirs = {}
for d in LOCAL_RAW.rglob('*'):
    if not d.is_dir():
        continue
    key = ann_group_key_from_name(d.name)
    if key is None:
        continue
    img_dir = find_child_dir(d, ['images', 'Images'])
    lab_dir = find_child_dir(d, ['labels', 'Labels'])
    if img_dir and lab_dir:
        n_img, n_lab = len(list_images(img_dir)), len(list_label_txts(lab_dir))
        if n_img > 0 and n_lab > 0:
            old = ann_group_dirs.get(key)
            if old is None or n_img > old['n_images']:
                ann_group_dirs[key] = {'root': d, 'images': img_dir, 'labels': lab_dir, 'n_images': n_img, 'n_labels': n_lab}

print('Annotated disease folders:')
for k in ['BG','WSSV','WSSV_BG']:
    print(k, '=>', ann_group_dirs.get(k))

missing_raw = {'Healthy','BG','WSSV','BG_WSSV'} - set(raw_class_dirs)
missing_ann = {'BG','WSSV','WSSV_BG'} - set(ann_group_dirs)
print('Missing raw:', missing_raw)
print('Missing annotated:', missing_ann)
if missing_ann:
    raise RuntimeError(f'Missing required annotated folders: {missing_ann}')

SyntaxError: unterminated string literal (detected at line 33) (3632451928.py, line 33)

## 5. Build raw image records and annotated label records

In [ ]:
raw_records, ann_image_records, box_records, label_errors = [], [], [], []

for cls, info in raw_class_dirs.items():
    for img_path in list_images(info['dir']):
        try:
            w, h = read_image_size(img_path); ok=True; err=''
        except Exception as e:
            w=h=None; ok=False; err=str(e)
        raw_records.append({'image_path':str(img_path),'file_name':img_path.name,'stem':img_path.stem,'raw_class':cls,
                            'width':w,'height':h,'aspect_ratio':(w/h) if w and h else None,
                            'file_size_kb':img_path.stat().st_size/1024,'read_ok':ok,'error':err})

for group, info in ann_group_dirs.items():
    img_dir, lab_dir = info['images'], info['labels']
    for img_path in list_images(img_dir):
        lab_path = lab_dir / f'{img_path.stem}.txt'
        try:
            w, h = read_image_size(img_path)
        except Exception as e:
            label_errors.append({'type':'image_read_error','image':str(img_path),'label':str(lab_path),'error':str(e)})
            continue
        if not lab_path.exists():
            label_errors.append({'type':'missing_label','image':str(img_path),'label':str(lab_path),'error':'missing label file'})
            continue
        text = lab_path.read_text().strip()
        source_classes, canonical_classes, n_boxes = [], [], 0
        if text:
            for line_no, line in enumerate(text.splitlines(), start=1):
                parts = line.split()
                if len(parts) != 5:
                    label_errors.append({'type':'malformed','image':str(img_path),'label':str(lab_path),'line':line_no,'error':line})
                    continue
                try:
                    src_cls = int(float(parts[0])); x,y,bw,bh = map(float, parts[1:])
                    can_cls = remap_label(group, src_cls)
                    valid_norm = (0 <= x <= 1 and 0 <= y <= 1 and 0 < bw <= 1 and 0 < bh <= 1)
                    x1=(x-bw/2)*w; y1=(y-bh/2)*h; x2=(x+bw/2)*w; y2=(y+bh/2)*h
                    inside = (x1>=0 and y1>=0 and x2<=w and y2<=h)
                    source_classes.append(src_cls); canonical_classes.append(can_cls); n_boxes += 1
                    box_records.append({'image_path':str(img_path),'label_path':str(lab_path),'file_name':img_path.name,'stem':img_path.stem,
                                        'source_group':group,'source_cls':src_cls,'canonical_cls_id':can_cls,
                                        'canonical_cls_name':'BG' if can_cls==0 else 'WSSV',
                                        'x_center_norm':x,'y_center_norm':y,'w_norm':bw,'h_norm':bh,
                                        'img_w':w,'img_h':h,'box_w_px':bw*w,'box_h_px':bh*h,
                                        'box_area_px2':bw*w*bh*h,'box_area_ratio':bw*bh,
                                        'x1_px':x1,'y1_px':y1,'x2_px':x2,'y2_px':y2,
                                        'valid_norm':valid_norm,'inside_image':inside,'line_no':line_no})
                    if not valid_norm or not inside:
                        label_errors.append({'type':'bbox_invalid_or_outside','image':str(img_path),'label':str(lab_path),'line':line_no,'error':line})
                except Exception as e:
                    label_errors.append({'type':'parse_error','image':str(img_path),'label':str(lab_path),'line':line_no,'error':str(e)})
        ann_image_records.append({'image_path':str(img_path),'label_path':str(lab_path),'file_name':img_path.name,'stem':img_path.stem,
                                  'source_group':group,'width':w,'height':h,'aspect_ratio':w/h if h else None,
                                  'file_size_kb':img_path.stat().st_size/1024,'n_boxes':n_boxes,
                                  'source_class_set':','.join(map(str,sorted(set(source_classes)))),
                                  'canonical_class_set':','.join(map(str,sorted(set(canonical_classes)))),
                                  'derived_image_level_from_boxes':image_level_from_label_set(set(canonical_classes))})

print('Raw image records:', len(raw_records))
print('Annotated image records:', len(ann_image_records))
print('Box records:', len(box_records))
print('Label errors:', len(label_errors))

def write_csv(path, rows):
    if not rows:
        Path(path).write_text(''); return
    fieldnames = sorted({k for r in rows for k in r.keys()})
    with Path(path).open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames); writer.writeheader(); writer.writerows(rows)

write_csv(TABLE_DIR/'raw_image_records.csv', raw_records)
write_csv(TABLE_DIR/'annotated_image_records.csv', ann_image_records)
write_csv(TABLE_DIR/'box_records_canonical.csv', box_records)
write_csv(TABLE_DIR/'label_errors.csv', label_errors)

if pd is not None:
    raw_df = pd.DataFrame(raw_records); ann_df = pd.DataFrame(ann_image_records); box_df = pd.DataFrame(box_records); err_df = pd.DataFrame(label_errors)
    display(raw_df.head()); display(ann_df.head()); display(box_df.head())
else:
    raw_df, ann_df, box_df, err_df = raw_records, ann_image_records, box_records, label_errors

## 6. Dataset summary tables

In [ ]:
if pd is not None:
    print('Raw classification class counts:')
    raw_counts = raw_df.groupby('raw_class').agg(images=('image_path','count'), mean_width=('width','mean'), mean_height=('height','mean'), mean_file_kb=('file_size_kb','mean')).reset_index()
    display(raw_counts); raw_counts.to_csv(TABLE_DIR/'summary_raw_class_counts.csv', index=False)

    print('
Annotated disease folder counts:')
    ann_counts = ann_df.groupby('source_group').agg(images=('image_path','count'), total_boxes=('n_boxes','sum'), mean_boxes_per_image=('n_boxes','mean'), median_boxes_per_image=('n_boxes','median'), max_boxes_per_image=('n_boxes','max')).reset_index()
    display(ann_counts); ann_counts.to_csv(TABLE_DIR/'summary_annotated_counts.csv', index=False)

    print('
Canonical box class counts:')
    box_counts = box_df.groupby('canonical_cls_name').agg(boxes=('image_path','count'), images_containing_class=('image_path','nunique'), mean_w_px=('box_w_px','mean'), median_w_px=('box_w_px','median'), mean_h_px=('box_h_px','mean'), median_h_px=('box_h_px','median'), mean_area_px2=('box_area_px2','mean'), median_area_px2=('box_area_px2','median')).reset_index()
    display(box_counts); box_counts.to_csv(TABLE_DIR/'summary_box_class_counts.csv', index=False)

    print('
Source-folder x canonical class counts:')
    cross = pd.crosstab(box_df['source_group'], box_df['canonical_cls_name'])
    display(cross); cross.to_csv(TABLE_DIR/'summary_source_by_canonical_class.csv')

    print('
Label errors:')
    if len(err_df): display(err_df.head(20))
    else: print('No label errors found.')
else:
    print('Install pandas to display detailed tables.')

## 7. Small-object analysis at 640, 1024, and 1280 image sizes

In [ ]:
def add_resized_box_stats(df, target_size):
    tmp = df.copy()
    scale = target_size / tmp[['img_w','img_h']].max(axis=1)
    tmp[f'box_w_px_at_{target_size}'] = tmp['box_w_px'] * scale
    tmp[f'box_h_px_at_{target_size}'] = tmp['box_h_px'] * scale
    tmp[f'box_area_px2_at_{target_size}'] = tmp[f'box_w_px_at_{target_size}'] * tmp[f'box_h_px_at_{target_size}']
    tmp[f'size_bucket_at_{target_size}'] = ['small(<32^2)' if a < 32*32 else ('medium(32^2-96^2)' if a < 96*96 else 'large(>=96^2)') for a in tmp[f'box_area_px2_at_{target_size}']]
    return tmp

if pd is not None and len(box_df):
    for sz in [640, 1024, 1280]:
        tmp = add_resized_box_stats(box_df, sz)
        bucket = pd.crosstab(tmp['canonical_cls_name'], tmp[f'size_bucket_at_{sz}'])
        print(f'
COCO-like box-size buckets at imgsz={sz}:')
        display(bucket)
        bucket.to_csv(TABLE_DIR/f'box_size_buckets_at_{sz}.csv')
else:
    print('Pandas unavailable or no boxes.')

## 8. Plot dataset distributions

In [ ]:
if pd is not None and plt is not None:
    plt.figure(figsize=(7,4)); raw_df['raw_class'].value_counts().reindex(['Healthy','BG','WSSV','BG_WSSV']).plot(kind='bar')
    plt.title('Raw image class distribution'); plt.ylabel('Images'); plt.tight_layout(); out=FIG_DIR/'raw_class_distribution.png'; plt.savefig(out,dpi=200); plt.show(); print('Saved:',out)

    plt.figure(figsize=(7,4)); ann_df['source_group'].value_counts().reindex(['BG','WSSV','WSSV_BG']).plot(kind='bar')
    plt.title('Annotated disease image distribution'); plt.ylabel('Images'); plt.tight_layout(); out=FIG_DIR/'annotated_image_distribution.png'; plt.savefig(out,dpi=200); plt.show(); print('Saved:',out)

    plt.figure(figsize=(6,4)); box_df['canonical_cls_name'].value_counts().reindex(['BG','WSSV']).plot(kind='bar')
    plt.title('Canonical bbox class distribution'); plt.ylabel('Boxes'); plt.tight_layout(); out=FIG_DIR/'canonical_box_class_distribution.png'; plt.savefig(out,dpi=200); plt.show(); print('Saved:',out)

    plt.figure(figsize=(8,4))
    for group in ['BG','WSSV','WSSV_BG']:
        vals=ann_df.loc[ann_df['source_group']==group,'n_boxes']; plt.hist(vals,bins=30,alpha=0.5,label=group)
    plt.title('Boxes per image by source group'); plt.xlabel('Number of boxes/image'); plt.ylabel('Images'); plt.legend(); plt.tight_layout(); out=FIG_DIR/'boxes_per_image_hist.png'; plt.savefig(out,dpi=200); plt.show(); print('Saved:',out)

    plt.figure(figsize=(8,4))
    for cls in ['BG','WSSV']:
        vals=box_df.loc[box_df['canonical_cls_name']==cls,'box_area_px2']; plt.hist(vals,bins=60,alpha=0.5,label=cls)
    plt.title('BBox area distribution at original resolution'); plt.xlabel('BBox area px^2'); plt.ylabel('Boxes'); plt.yscale('log'); plt.legend(); plt.tight_layout(); out=FIG_DIR/'bbox_area_distribution_original.png'; plt.savefig(out,dpi=200); plt.show(); print('Saved:',out)

    plt.figure(figsize=(6,6))
    for cls in ['BG','WSSV']:
        sub=box_df[box_df['canonical_cls_name']==cls]; plt.scatter(sub['box_w_px'],sub['box_h_px'],s=8,alpha=0.35,label=cls)
    plt.title('BBox width vs height at original resolution'); plt.xlabel('Box width px'); plt.ylabel('Box height px'); plt.legend(); plt.tight_layout(); out=FIG_DIR/'bbox_width_height_scatter.png'; plt.savefig(out,dpi=200); plt.show(); print('Saved:',out)
else:
    print('Skipping plots because pandas or matplotlib is unavailable.')

## 9. BBox center heatmaps

In [ ]:
if pd is not None and plt is not None and np is not None and len(box_df) > 0:
    for cls in ['BG','WSSV']:
        sub=box_df[box_df['canonical_cls_name']==cls]
        heat, xedges, yedges = np.histogram2d(sub['x_center_norm'], sub['y_center_norm'], bins=40, range=[[0,1],[0,1]])
        plt.figure(figsize=(6,5)); plt.imshow(heat.T, origin='lower', extent=[0,1,0,1], aspect='auto')
        plt.colorbar(label='Box centers'); plt.title(f'BBox center heatmap: {cls}'); plt.xlabel('x center normalized'); plt.ylabel('y center normalized')
        out=FIG_DIR/f'bbox_center_heatmap_{cls}.png'; plt.savefig(out,dpi=200,bbox_inches='tight'); plt.show(); print('Saved:',out)
else:
    print('Skipping heatmaps.')

## 10. Visualization helpers: overlays and contact sheets

In [ ]:
COLOR_MAP = {0:(255,60,60), 1:(60,180,255)}
CLASS_NAME = {0:'BG', 1:'WSSV'}

def load_font(size=14):
    try: return ImageFont.truetype('DejaVuSans.ttf', size)
    except Exception: return ImageFont.load_default()

def draw_boxes_on_image(img_path, label_path, source_group, max_side=900):
    im = Image.open(img_path).convert('RGB'); w,h = im.size; draw = ImageDraw.Draw(im); font = load_font(18)
    if label_path and Path(label_path).exists():
        text=Path(label_path).read_text().strip()
        if text:
            for line in text.splitlines():
                parts=line.split()
                if len(parts)!=5: continue
                src_cls=int(float(parts[0])); x,y,bw,bh=map(float,parts[1:]); can_cls=remap_label(source_group,src_cls)
                x1=(x-bw/2)*w; y1=(y-bh/2)*h; x2=(x+bw/2)*w; y2=(y+bh/2)*h
                color=COLOR_MAP.get(can_cls,(255,255,0)); draw.rectangle([x1,y1,x2,y2], outline=color, width=max(3,int(w/700)))
                draw.text((x1+3,max(0,y1-22)), CLASS_NAME.get(can_cls,str(can_cls)), fill=color, font=font)
    scale=min(max_side/max(w,h),1.0)
    if scale<1.0: im=im.resize((int(w*scale),int(h*scale)))
    return im

def make_contact_sheet(image_paths, labels=None, title='', thumb_size=(160,160), cols=5, bg=(255,255,255)):
    labels=labels or ['']*len(image_paths); rows=math.ceil(len(image_paths)/cols); title_h=34 if title else 0
    cell_w, cell_h=thumb_size[0], thumb_size[1]+34
    sheet=Image.new('RGB',(cols*cell_w, rows*cell_h+title_h), bg); draw=ImageDraw.Draw(sheet); font=load_font(12); title_font=load_font(18)
    if title: draw.text((10,8), title, fill=(0,0,0), font=title_font)
    for i,path in enumerate(image_paths):
        r,c=divmod(i,cols); x0=c*cell_w; y0=r*cell_h+title_h
        try:
            im=Image.open(path).convert('RGB'); im.thumbnail(thumb_size); sheet.paste(im,(x0+(thumb_size[0]-im.width)//2, y0+(thumb_size[1]-im.height)//2))
        except Exception:
            draw.text((x0+5,y0+5),'ERR',fill=(255,0,0),font=font)
        draw.text((x0+4,y0+thumb_size[1]+4), str(labels[i])[:22], fill=(0,0,0), font=font)
    return sheet

def make_overlay_contact_sheet(records, title='', sample_n=24, cols=4, seed=42):
    items=list(records); random.Random(seed).shuffle(items); items=items[:sample_n]
    thumbs=[]; labels=[]
    for r in items:
        im=draw_boxes_on_image(r['image_path'], r['label_path'], r['source_group'], max_side=600); im.thumbnail((220,220)); thumbs.append(im.copy())
        labels.append(f"{r['source_group']} | {Path(r['image_path']).name}")
    rows=math.ceil(len(thumbs)/cols); cell_w,cell_h=240,265; title_h=34
    sheet=Image.new('RGB',(cols*cell_w, rows*cell_h+title_h),(255,255,255)); draw=ImageDraw.Draw(sheet); font=load_font(11); title_font=load_font(18)
    draw.text((10,8),title,fill=(0,0,0),font=title_font)
    for i,im in enumerate(thumbs):
        r,c=divmod(i,cols); x0,y0=c*cell_w, r*cell_h+title_h; sheet.paste(im,(x0+(220-im.width)//2,y0+(220-im.height)//2)); draw.text((x0+4,y0+224),labels[i][:32],fill=(0,0,0),font=font)
    return sheet

## 11. Raw image contact sheets by class

In [ ]:
for cls in ['Healthy','BG','WSSV','BG_WSSV']:
    info=raw_class_dirs.get(cls)
    if not info: continue
    imgs=list_images(info['dir']); sample=random.sample(imgs, min(30,len(imgs))); labels=[p.name for p in sample]
    sheet=make_contact_sheet(sample, labels, title=f'Raw samples: {cls}', thumb_size=(160,160), cols=6)
    out=GALLERY_DIR/f'raw_samples_{cls}.jpg'; sheet.save(out, quality=92); display(sheet); print('Saved:',out)

## 12. Annotated overlay samples by disease group

In [ ]:
if pd is not None:
    for group in ['BG','WSSV','WSSV_BG']:
        sub=ann_df[ann_df['source_group']==group].to_dict('records')
        sheet=make_overlay_contact_sheet(sub, title=f'Annotated overlay samples: {group}', sample_n=24, cols=4, seed=SEED)
        out=GALLERY_DIR/f'overlay_samples_{group}.jpg'; sheet.save(out, quality=92); display(sheet); print('Saved:',out)
else:
    print('Pandas unavailable.')

## 13. Full dataset thumbnail contact sheets

In [ ]:
def save_paged_contact_sheets(image_paths, labels, prefix, title_prefix, per_page=80, cols=8, thumb_size=(120,120)):
    outputs=[]
    for start in range(0,len(image_paths),per_page):
        page_paths=image_paths[start:start+per_page]; page_labels=labels[start:start+per_page]; page_idx=start//per_page+1
        sheet=make_contact_sheet(page_paths,page_labels,title=f'{title_prefix} — page {page_idx}',thumb_size=thumb_size,cols=cols)
        out=GALLERY_DIR/f'{prefix}_page_{page_idx:03d}.jpg'; sheet.save(out,quality=88); outputs.append(out)
    return outputs

for cls in ['Healthy','BG','WSSV','BG_WSSV']:
    info=raw_class_dirs.get(cls)
    if not info: continue
    imgs=list_images(info['dir']); labels=[p.name for p in imgs]
    outs=save_paged_contact_sheets(imgs, labels, prefix=f'FULL_raw_{cls}', title_prefix=f'FULL raw {cls}', per_page=80)
    print(cls, 'pages:', len(outs), 'first:', outs[0] if outs else None)

if pd is not None:
    for group in ['BG','WSSV','WSSV_BG']:
        sub=ann_df[ann_df['source_group']==group].to_dict('records')
        overlay_paths=[]; overlay_labels=[]; group_overlay_dir=OVERLAY_DIR/group; group_overlay_dir.mkdir(parents=True,exist_ok=True)
        for r in sub:
            out=group_overlay_dir/f"{Path(r['image_path']).stem}_overlay.jpg"
            if not out.exists():
                im=draw_boxes_on_image(r['image_path'], r['label_path'], r['source_group'], max_side=700); im.save(out,quality=90)
            overlay_paths.append(out); overlay_labels.append(Path(r['image_path']).name)
        outs=save_paged_contact_sheets(overlay_paths, overlay_labels, prefix=f'FULL_overlay_{group}', title_prefix=f'FULL overlay {group}', per_page=48, cols=6, thumb_size=(150,150))
        print(group, 'overlay pages:', len(outs), 'first:', outs[0] if outs else None)

print('Gallery directory:', GALLERY_DIR)
print('Overlay directory:', OVERLAY_DIR)

## 14. Optional: analyze prior YOLO benchmark results if available

In [ ]:
candidate_result_paths = [
    Path('/home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v3_FIXED_OD_ONLY/tables/final_model_comparison.csv'),
    Path('/home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v2/tables/final_model_comparison.csv'),
    WORKDIR.parent / 'shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v3_FIXED_OD_ONLY' / 'tables' / 'final_model_comparison.csv',
]
found_result=None
for p in candidate_result_paths:
    if p.exists(): found_result=p; break

if found_result is None:
    print('No previous model comparison CSV found. Skipping model-result analysis.')
elif pd is None:
    print('Found CSV but pandas unavailable:', found_result)
else:
    print('Found model comparison:', found_result)
    mdf=pd.read_csv(found_result); display(mdf.head()); mdf.to_csv(TABLE_DIR/'imported_model_comparison.csv',index=False)
    ok=mdf[mdf['status']=='ok'].copy() if 'status' in mdf.columns else mdf.copy()
    rank_cols=[c for c in ['model','map50','map50_95','precision_mp','recall_mr','diagnosis_macro_f1_disease_only','disease_miss_rate_no_det','fps_total','model_size_mb','params_m','mobile_edge_score'] if c in ok.columns]
    if 'mobile_edge_score' in ok.columns: ok=ok.sort_values('mobile_edge_score',ascending=False)
    elif 'map50_95' in ok.columns: ok=ok.sort_values('map50_95',ascending=False)
    display(ok[rank_cols]); ok[rank_cols].to_csv(TABLE_DIR/'ranked_model_results_subset.csv',index=False)
    if plt is not None and len(ok):
        if 'fps_total' in ok.columns and 'map50_95' in ok.columns:
            plt.figure(figsize=(8,5)); plt.scatter(ok['fps_total'],ok['map50_95'])
            for _,r in ok.iterrows(): plt.annotate(str(r['model']),(r['fps_total'],r['map50_95']),fontsize=8)
            plt.xlabel('FPS'); plt.ylabel('mAP50-95'); plt.title('Model benchmark: mAP50-95 vs FPS'); plt.grid(True,alpha=0.3); out=FIG_DIR/'model_map50_95_vs_fps.png'; plt.savefig(out,dpi=200,bbox_inches='tight'); plt.show(); print('Saved:',out)
        if 'diagnosis_macro_f1_disease_only' in ok.columns and 'model_size_mb' in ok.columns:
            plt.figure(figsize=(8,5)); plt.scatter(ok['model_size_mb'],ok['diagnosis_macro_f1_disease_only'])
            for _,r in ok.iterrows(): plt.annotate(str(r['model']),(r['model_size_mb'],r['diagnosis_macro_f1_disease_only']),fontsize=8)
            plt.xlabel('Model size MB'); plt.ylabel('Disease-only diagnosis Macro-F1'); plt.title('Model benchmark: diagnosis Macro-F1 vs model size'); plt.grid(True,alpha=0.3); out=FIG_DIR/'model_diag_f1_vs_size.png'; plt.savefig(out,dpi=200,bbox_inches='tight'); plt.show(); print('Saved:',out)

## 15. Generate EDA markdown report

In [ ]:
report_path=EDA_DIR/'EDA_REPORT.md'
report=[]
report.append('# ShrimpDiseaseImageBD EDA Report')
report.append('')
report.append(f'Generated: {datetime.now().isoformat(timespec="seconds")}')
report.append('')
report.append('## Dataset source')
report.append(f'- Kaggle dataset: `{KAGGLE_DATASET}`')
report.append(f'- Local raw path: `{LOCAL_RAW}`')
report.append('')
report.append('## Raw class folders')
for cls in ['Healthy','BG','WSSV','BG_WSSV']:
    v=raw_class_dirs.get(cls); report.append(f'- {cls}: {v["n_images"] if v else 0} images | `{v["dir"] if v else "missing"}`')
report.append('')
report.append('## Annotated disease folders')
for group in ['BG','WSSV','WSSV_BG']:
    v=ann_group_dirs.get(group); report.append(f'- {group}: {v["n_images"] if v else 0} images, {v["n_labels"] if v else 0} labels | `{v["root"] if v else "missing"}`')
report.append('')
report.append('## Canonical OD mapping')
report.append('- `0 = BG`')
report.append('- `1 = WSSV`')
report.append('- `BG/source class 0 -> BG`')
report.append('- `WSSV/source class 0 -> WSSV`')
report.append('- `WSSV_BG/source class 0 -> WSSV`, `WSSV_BG/source class 1 -> BG`')
report.append('')
report.append('## Main warnings')
report.append(f'- Label errors found: {len(label_errors)}. See `tables/label_errors.csv`.' if label_errors else '- No malformed/out-of-range labels found by this audit.')
report.append('- Healthy has no disease-region boxes and should not be treated as a box-level OD class.')
report.append('- For final mobile system, use a Healthy classification gate before the disease detector.')
report.append('- For final paper, report both box-level OD metrics and image-level diagnosis metrics.')
report.append('')
report.append('## Output folders')
report.append(f'- Figures: `{FIG_DIR}`')
report.append(f'- Tables: `{TABLE_DIR}`')
report.append(f'- Galleries: `{GALLERY_DIR}`')
report.append(f'- Overlays: `{OVERLAY_DIR}`')
report_path.write_text('\n'.join(report))
print('Saved report:', report_path)
print('\n'.join(report[:45]))